# Notebook 08 — Phase-Matched CZ Optimization

This notebook searches for **candidate gate settings** where three conditions are satisfied together:

1. the effective operation is close to diagonal,
2. the entangling phase is close to $\pi$,
3. the overall gate is close to a controlled-phase operation.

Rather than relying on raw process fidelity alone, we define a simple composite score that rewards:

- high phase-corrected fidelity,
- low off-diagonal leakage,
- entangling phase near $\pi$.

This is the natural next step after Notebook 07: **synchronize phase and population return**.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, sigmay, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
sy = sigmay()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
sy1 = tensor(sy, I)
sy2 = tensor(I, sy)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Hamiltonian and pulse-sequence helpers

In [ ]:
def two_atom_hamiltonian(omega1: float, omega2: float, delta: float, V: float, axis: str = 'x'):
    if axis == 'x':
        drive = 0.5 * omega1 * sx1 + 0.5 * omega2 * sx2
    elif axis == 'y':
        drive = 0.5 * omega1 * sy1 + 0.5 * omega2 * sy2
    else:
        raise ValueError("axis must be 'x' or 'y'")
    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction

def run_segment(state, omega1: float, omega2: float, delta: float, V: float,
                duration: float, axis: str = 'x', n_steps: int = 140):
    H = two_atom_hamiltonian(omega1=omega1, omega2=omega2, delta=delta, V=V, axis=axis)
    times = np.linspace(0.0, duration, n_steps)
    result = sesolve(H, state, times)
    return result.states[-1]

def run_minimal_sequence(psi0, omega: float, delta: float, V: float,
                         t_pi: float, t_mid: float,
                         axis1: str = 'x', axis2: str = 'y', axis3: str = 'x'):
    state1 = run_segment(psi0, omega1=omega, omega2=0.0, delta=delta, V=V, duration=t_pi, axis=axis1)
    state2 = run_segment(state1, omega1=0.0, omega2=omega, delta=delta, V=V, duration=t_mid, axis=axis2)
    state3 = run_segment(state2, omega1=omega, omega2=0.0, delta=delta, V=V, duration=t_pi, axis=axis3)
    return state3

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)

def build_effective_unitary(omega: float, delta: float, V: float, t_pi: float, t_mid: float,
                            axis1: str = 'x', axis2: str = 'y', axis3: str = 'x'):
    columns = []
    for psi0 in basis_states:
        psi_final = run_minimal_sequence(
            psi0, omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid,
            axis1=axis1, axis2=axis2, axis3=axis3
        )
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Gate metrics

In [ ]:
def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, U_target=phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))

def phase_distance_to_pi(U_eff):
    return float(np.abs(np.abs(entangling_phase(U_eff)) - np.pi) / np.pi)

def composite_score(U_eff, w_phase=0.45, w_fid=0.40, w_leak=0.15):
    pc_fid = phase_corrected_fidelity(U_eff)
    leak = leakage_metric(U_eff)
    phase_dist = phase_distance_to_pi(U_eff)
    leak_penalty = np.exp(-4.0 * leak)
    phase_reward = np.exp(-6.0 * phase_dist)
    return float(w_phase * phase_reward + w_fid * pc_fid + w_leak * leak_penalty)


## Coarse search over $(\Omega, V, t_{mid})$

In [ ]:
omega_vals = np.linspace(0.6, 1.8, 13)
V_vals = np.linspace(0.5, 6.0, 12)
tmid_factors = np.linspace(0.8, 3.0, 23)  # t_mid = factor * pi / omega

results = []
for omega in omega_vals:
    t_pi = np.pi / omega
    for V in V_vals:
        for fac in tmid_factors:
            t_mid = fac * np.pi / omega
            U_eff = build_effective_unitary(
                omega=omega, delta=0.0, V=V, t_pi=t_pi, t_mid=t_mid,
                axis1='x', axis2='y', axis3='x'
            )
            results.append({
                'omega': float(omega),
                'V': float(V),
                't_pi': float(t_pi),
                't_mid': float(t_mid),
                'tmid_factor': float(fac),
                'raw_fid': float(process_fidelity(U_eff)),
                'pc_fid': float(phase_corrected_fidelity(U_eff)),
                'phi_ent': float(entangling_phase(U_eff)),
                'phi_ent_over_pi': float(entangling_phase(U_eff) / np.pi),
                'phase_dist': float(phase_distance_to_pi(U_eff)),
                'leak': float(leakage_metric(U_eff)),
                'score': float(composite_score(U_eff)),
            })

results = sorted(results, key=lambda x: x['score'], reverse=True)
top10 = results[:10]
top10[:3]


## Top candidate settings

In [ ]:
for i, r in enumerate(top10, start=1):
    print(
        f"{i:02d}. "
        f"Omega={r['omega']:.3f}, V={r['V']:.3f}, t_mid={r['t_mid']:.3f} ({r['tmid_factor']:.3f} * pi/Omega), "
        f"score={r['score']:.4f}, raw={r['raw_fid']:.4f}, pc={r['pc_fid']:.4f}, "
        f"phi/pi={r['phi_ent_over_pi']:.4f}, leak={r['leak']:.4f}"
    )


## Inspect the best candidate

In [ ]:
best = top10[0]
U_best = build_effective_unitary(
    omega=best['omega'], delta=0.0, V=best['V'],
    t_pi=best['t_pi'], t_mid=best['t_mid'],
    axis1='x', axis2='y', axis3='x'
)

print('Best candidate metrics:')
print(best)

plt.figure(figsize=(6, 5))
im = plt.imshow(np.abs(U_best), origin='lower', aspect='equal')
plt.xticks(range(4), basis_labels)
plt.yticks(range(4), basis_labels)
plt.xlabel('Input basis state')
plt.ylabel('Output basis amplitude')
plt.title(r'$|U_{eff}|$ for best candidate')
plt.colorbar(im, label='magnitude')
plt.tight_layout()
plt.show()


## Best-score slices over $t_{mid}$ for a fixed $(\Omega, V)$

In [ ]:
omega = best['omega']
V = best['V']
t_pi = np.pi / omega
t_mid_vals = np.linspace(0.6 * np.pi / omega, 3.2 * np.pi / omega, 60)

raw_vals = []
pc_vals = []
phi_vals = []
leak_vals = []
score_vals = []

for t_mid in t_mid_vals:
    U_eff = build_effective_unitary(
        omega=omega, delta=0.0, V=V, t_pi=t_pi, t_mid=t_mid,
        axis1='x', axis2='y', axis3='x'
    )
    raw_vals.append(process_fidelity(U_eff))
    pc_vals.append(phase_corrected_fidelity(U_eff))
    phi_vals.append(entangling_phase(U_eff) / np.pi)
    leak_vals.append(leakage_metric(U_eff))
    score_vals.append(composite_score(U_eff))


In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, score_vals, label='composite score')
plt.plot(t_mid_vals, pc_vals, label='phase-corrected fidelity')
plt.plot(t_mid_vals, raw_vals, label='raw fidelity')
plt.axvline(best['t_mid'], linestyle='--', label='best candidate')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel('Score / fidelity')
plt.title('Optimization slice at the best $(\Omega, V)$')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, phi_vals, label=r'$\phi_{ent}/\pi$')
plt.axhline(1.0, linestyle='--', label='ideal +1')
plt.axhline(-1.0, linestyle='--', color='gray', label='ideal -1')
plt.axvline(best['t_mid'], linestyle='--')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel(r'Entangling phase / $\pi$')
plt.title('Entangling phase near the best candidate')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, leak_vals)
plt.axvline(best['t_mid'], linestyle='--')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel('Leakage norm')
plt.title('Leakage near the best candidate')
plt.tight_layout()
plt.show()


## 2D optimization map at the best $t_{mid}$ factor

In [ ]:
best_factor = best['tmid_factor']
omega_grid = np.linspace(0.6, 1.8, 22)
V_grid = np.linspace(0.5, 6.0, 22)

score_map = np.zeros((len(V_grid), len(omega_grid)))
pc_map = np.zeros((len(V_grid), len(omega_grid)))
phi_map = np.zeros((len(V_grid), len(omega_grid)))

for i, V_scan in enumerate(V_grid):
    for j, omega_scan in enumerate(omega_grid):
        t_pi_scan = np.pi / omega_scan
        t_mid_scan = best_factor * np.pi / omega_scan
        U_eff = build_effective_unitary(
            omega=omega_scan, delta=0.0, V=V_scan,
            t_pi=t_pi_scan, t_mid=t_mid_scan,
            axis1='x', axis2='y', axis3='x'
        )
        score_map[i, j] = composite_score(U_eff)
        pc_map[i, j] = phase_corrected_fidelity(U_eff)
        phi_map[i, j] = entangling_phase(U_eff) / np.pi


In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(score_map, origin='lower', aspect='auto', extent=[omega_grid[0], omega_grid[-1], V_grid[0], V_grid[-1]])
plt.contour(omega_grid, V_grid, score_map, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Composite optimization score over $(\Omega, V)$')
plt.colorbar(im, label='score')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(pc_map, origin='lower', aspect='auto', extent=[omega_grid[0], omega_grid[-1], V_grid[0], V_grid[-1]])
plt.contour(omega_grid, V_grid, pc_map, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Phase-corrected fidelity over $(\Omega, V)$')
plt.colorbar(im, label='phase-corrected fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(phi_map, origin='lower', aspect='auto', extent=[omega_grid[0], omega_grid[-1], V_grid[0], V_grid[-1]], vmin=-1, vmax=1)
plt.contour(omega_grid, V_grid, phi_map, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Entangling phase / $\pi$ over $(\Omega, V)$ at best factor')
plt.colorbar(im, label=r'$\phi_{ent}/\pi$')
plt.tight_layout()
plt.show()


## Interpretation

This notebook upgrades the workflow from diagnosis to optimization.

The main idea is that a usable controlled-phase setting must satisfy **all three**:

- entangling phase near $\pi$,
- low leakage,
- strong overlap with a diagonal controlled-phase target.

A good candidate can still have low **raw** CZ fidelity if the diagonal phases are not fully calibrated, while still showing strong **phase-corrected** fidelity. That is useful because it distinguishes:

1. a fundamentally wrong gate,
2. an almost-correct gate with removable phase offsets,
3. a phase-correct gate spoiled by leakage.


## Optional next steps

Natural next upgrades are:

- refine the search around the top candidates with a denser local grid,
- add detuning as an optimization variable,
- include explicit single-qubit phase corrections,
- move from grid search to numerical optimal control.
